[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week07.ipynb)

# Week 7: Regularization & Generalization
**Introduction to Deep Learning (HIT)** &middot; Part II: Training Infrastructure

Overfitting; dropout, weight decay, early stopping; basic data augmentation.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Diagnose overfitting from the train and validation gap.
- Apply dropout, weight decay, early stopping, and augmentation.
- Attribute a generalization gain to a specific cause.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. Make a model overfit
Tiny training set, large model: train accuracy goes high, test stays at chance.

In [ ]:
torch.manual_seed(0)
Xtr = torch.randn(30, 20); ytr = torch.randint(0, 2, (30,))   # 30 samples, random labels
Xte = torch.randn(300, 20); yte = torch.randint(0, 2, (300,))

def fit(model, epochs=250, wd=0.0):
    o = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=wd); f = nn.CrossEntropyLoss()
    tr, te = [], []
    for _ in range(epochs):
        model.train(); o.zero_grad(); f(model(Xtr), ytr).backward(); o.step()
        model.eval()
        with torch.no_grad():
            tr.append((model(Xtr).argmax(1) == ytr).float().mean().item())
            te.append((model(Xte).argmax(1) == yte).float().mean().item())
    return tr, te

big = lambda: nn.Sequential(nn.Linear(20, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
tr, te = fit(big())
print(f"overfit: train {tr[-1]:.2f} vs test {te[-1]:.2f}")
plt.plot(tr, label="train"); plt.plot(te, label="test"); plt.legend(); plt.title("Overfitting"); plt.show()

## 2. Weight decay and dropout close the gap
Same model, add regularization, watch the gap shrink.

In [ ]:
reg = nn.Sequential(nn.Linear(20, 128), nn.ReLU(), nn.Dropout(0.5),
                    nn.Linear(128, 128), nn.ReLU(), nn.Dropout(0.5), nn.Linear(128, 2))
tr, te = fit(reg, wd=1e-2)
print(f"regularized: train {tr[-1]:.2f} vs test {te[-1]:.2f}")
plt.plot(tr, label="train"); plt.plot(te, label="test"); plt.legend(); plt.title("With dropout + weight decay"); plt.show()

## 3. Dropout behaves differently at train vs eval
On in training (random zeros), off at eval (deterministic).

In [ ]:
drop = nn.Dropout(0.5)
x = torch.ones(1, 10)
drop.train(); print("train (random zeros):", drop(x))
drop.eval();  print("eval  (identity)    :", drop(x))

## 4. Data augmentation (train only)
Augmentation enlarges the effective training set.

In [ ]:
from torchvision import transforms
img = torch.rand(3, 32, 32)
aug = transforms.Compose([transforms.RandomHorizontalFlip(p=1.0), transforms.RandomCrop(32, padding=4)])
print("original", tuple(img.shape), "-> augmented", tuple(aug(img).shape))
print("two augmentations differ:", not torch.equal(aug(img), aug(img)))

**Discuss:** why must augmentation and dropout be OFF at test time? (We evaluate the real model on real inputs.)

---
Student materials for this week: the lab handout (`labs/week07.html`) and the curated references (`references/week07.html`) in the course site.